# Hands-on: Pandora, TEMPO and ABI in Python

Companion notebook to *Seeing Smoke and Dust from Geostationary Orbit*. It downloads Pandora and TEMPO NO₂ for one evening over Austin, filters and matches them, then reads TEMPO's aerosol products and NOAA's ABI AOD.

Steps 2 and 4–6 have been run on real files; step 3 needs a free NASA Earthdata login. Steps 7–9 follow the products' user guides; the TEMPO aerosol products only become generally available around spring 2027, so those searches may return nothing yet.

## 1. Set up

Install the tools. In Google Colab, run this once per session. pandora-tools is the reader for Pandora files; earthaccess handles NASA Earthdata downloads (you need a free Earthdata account).

In [ ]:
!pip install -q git+https://github.com/paulsavala/pandora-tools earthaccess xarray netCDF4 h5netcdf pyproj s3fs
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
import pandora_tools as pdt

## 2. Pandora NO₂ at St. Edward's

Download Pandora 257's sky-scan NO₂ (product rnvh3) for the evening of the scan used on this site. Keep only high-quality readings (flags 0 and 10) and convert moles/m² to molecules/cm², TEMPO's unit.

In [ ]:
client = pdt.PandoraClient()
pandora = client.fetch("AustinTX", 257, "2026-08-27", "2026-08-28", products="rnvh3")
good = pandora[pandora["no2_qa"].isin([0, 10])]        # assured or not-assured HIGH quality only
# Pandora reports moles per m²; TEMPO reports molecules per cm². 1 mol/m² = 6.022e19 molecules/cm².
good = good.assign(no2_trop=good["no2_trop_col"] * 6.022e19)
lat0, lon0 = float(good["latitude"].iloc[0]), float(good["longitude"].iloc[0])
good.plot(x="local_time", y="no2_trop", style=".", ylabel="NO₂ tropospheric column (molecules/cm²)")

## 3. Find and download TEMPO NO₂ granules

Search NASA's catalog for TEMPO NO₂ Level 2 files over the site for the same day, in UTC (see “A Day of Measurements”).

*Needs a free NASA Earthdata login.*

In [ ]:
import earthaccess
earthaccess.login()                                     # free NASA Earthdata account
results = earthaccess.search_data(short_name="TEMPO_NO2_L2", version="V04",
                                  temporal=("2026-08-27 13:00", "2026-08-28 02:00"),
                                  bounding_box=(lon0 - 0.5, lat0 - 0.5, lon0 + 0.5, lat0 + 0.5))
paths = earthaccess.download(results, "tempo_files")

## 4. Open a TEMPO file as a table

Read the three groups described in “Inside a TEMPO File” and flatten the mirror-step × cross-track grid into one row per pixel.

In [ ]:
def open_tempo_no2(path):
    """One TEMPO NO2 L2 granule as a flat table: one row per pixel."""
    prod = xr.open_dataset(path, group="product")
    geo = xr.open_dataset(path, group="geolocation")
    sup = xr.open_dataset(path, group="support_data")
    table = pd.DataFrame({
        "lat": geo["latitude"].values.ravel(),
        "lon": geo["longitude"].values.ravel(),
        "time": np.repeat(geo["time"].values, geo.sizes["xtrack"]),   # one time per mirror step
        "no2_trop": prod["vertical_column_troposphere"].values.ravel(),
        "flag": prod["main_data_quality_flag"].values.ravel(),
        "cloud": sup["eff_cloud_fraction"].values.ravel(),
    })
    table["time"] = pd.to_datetime(table["time"]).dt.tz_localize("UTC")
    return table.dropna(subset=["no2_trop"])

tempo = pd.concat([open_tempo_no2(p) for p in paths], ignore_index=True)
print(tempo["flag"].value_counts())                     # 0 normal, 1 suspicious, 2 bad

## 5. Filter by quality and clouds

Normal-quality pixels only, then a cloud-fraction cutoff. Print how many survive.

In [ ]:
clean = tempo[(tempo["flag"] == 0) & (tempo["cloud"] < 0.2)]
print(f"kept {len(clean)} of {len(tempo)} pixels")

## 6. Match Pandora to TEMPO

For each scan, average the pixels within 3 km of the site and the Pandora readings within ±30 minutes of the pixels' own time (see “Matching Instruments in Space and Time”).

In [ ]:
def nearest_pixels(table, lat, lon, max_km=3):
    """Pixels whose center is within max_km of a site (TEMPO pixels are ~2 x 5 km)."""
    dx = (table["lon"] - lon) * 111.32 * np.cos(np.radians(lat))
    dy = (table["lat"] - lat) * 110.57
    return table[np.hypot(dx, dy) <= max_km]

near = nearest_pixels(clean, lat0, lon0)
pairs = []
for scan_time, pix in near.groupby(near["time"].dt.floor("10min")):
    t = pix["time"].mean()
    window = good[(good["utc_time"] - t).abs() <= pd.Timedelta("30min")]   # Pandora within ±30 min
    if len(window):
        pairs.append({"time": t, "tempo": pix["no2_trop"].mean(), "pandora": window["no2_trop"].mean(), "n_pandora": len(window)})
pairs = pd.DataFrame(pairs)
pairs.plot.scatter(x="pandora", y="tempo")

## 7. TEMPO AOD and layer height

TEMPO's combined aerosol product (TEMPO_AODALH_L2). Keep high-quality AOD for quantitative work; layer height exists only where AOD is above 0.2.

*Written from the product user guide; not yet run on real files.*

In [ ]:
aod_results = earthaccess.search_data(short_name="TEMPO_AODALH_L2",
                                      temporal=("2026-08-27 13:00", "2026-08-28 02:00"),
                                      bounding_box=(lon0 - 0.5, lat0 - 0.5, lon0 + 0.5, lat0 + 0.5))
aod_paths = earthaccess.download(aod_results, "tempo_aod")

def open_tempo_aod(path):
    """TEMPO AOD (550 nm) and aerosol layer height, one row per pixel."""
    prod = xr.open_dataset(path, group="product")
    geo = xr.open_dataset(path, group="geolocation")
    qf = xr.open_dataset(path, group="quality_diagnostic_flags")
    return pd.DataFrame({
        "lat": geo["latitude"].values.ravel(), "lon": geo["longitude"].values.ravel(),
        "aod550": prod["aod550"].values.ravel(),
        "alh_km": prod["alh"].values.ravel(),
        "dqf": qf["dqf"].values.ravel(),          # 0 high, 1 medium, 2 low, 3 no retrieval
    }).dropna(subset=["aod550"])

aod = pd.concat([open_tempo_aod(p) for p in aod_paths], ignore_index=True)
aod_q = aod[(aod["dqf"] == 0) & (aod["aod550"] <= 5)]     # quantitative use: high quality, no extreme values
alh_q = aod_q[aod_q["aod550"] > 0.2]                      # layer height is only retrieved where AOD > 0.2

## 8. TEMPO's smoke/dust mask

The TEMPO-ABI aerosol detection product (TEMPOABI_ADP_L2): smoke and dust flags, the UV aerosol index, and confidence stored as bits in qc_flag. For quantitative use, keep high and medium confidence.

*Written from the product user guide; not yet run on real files.*

In [ ]:
adp_results = earthaccess.search_data(short_name="TEMPOABI_ADP_L2",
                                      temporal=("2026-08-27 13:00", "2026-08-28 02:00"),
                                      bounding_box=(lon0 - 0.5, lat0 - 0.5, lon0 + 0.5, lat0 + 0.5))
adp_paths = earthaccess.download(adp_results, "tempo_adp")

def open_tempo_adp(path):
    """TEMPO-ABI smoke/dust mask, UV aerosol index and confidence, one row per pixel."""
    prod = xr.open_dataset(path, group="product")
    geo = xr.open_dataset(path, group="geolocation")
    qc = xr.open_dataset(path, group="quality_diagnostic_flags")["qc_flag"].values.ravel().astype("int64")
    return pd.DataFrame({
        "lat": geo["latitude"].values.ravel(), "lon": geo["longitude"].values.ravel(),
        "smoke": prod["smoke"].values.ravel(), "dust": prod["dust"].values.ravel(),
        "uv_aai": prod["uv_aai"].values.ravel(),
        "smoke_conf": qc & 0b001100,              # 0 high, 4 medium, 8 low
        "dust_conf": qc & 0b110000,               # 0 high, 16 medium, 32 low
    })

adp = pd.concat([open_tempo_adp(p) for p in adp_paths], ignore_index=True)
smoke_q = adp[(adp["smoke"] == 1) & adp["smoke_conf"].isin([0, 4])]   # quantitative: high + medium confidence
plt.scatter(adp["lon"], adp["lat"], c=adp["uv_aai"], s=1, cmap="magma"); plt.colorbar(label="UV aerosol index")
plt.scatter(smoke_q["lon"], smoke_q["lat"], s=1, c="k"); plt.title("Smoke (high + medium confidence) over the UV aerosol index")

## 9. ABI AOD from NOAA's public cloud

GOES-19 ABI AOD for the continental US, read straight from NOAA's open bucket on Amazon Web Services with no login. ABI data sit on a fixed grid of scan angles, so latitude and longitude are computed from the projection.

*Written from the product user guide; not yet run on real files.*

In [ ]:
import s3fs
from pyproj import Proj
fs = s3fs.S3FileSystem(anon=True)                         # NOAA's public bucket: no login needed
files = fs.ls("noaa-goes19/ABI-L2-AODC/2026/239/19/")      # year / day of year (Aug 27 = 239) / hour, UTC
with fs.open(files[0]) as f:
    abi = xr.open_dataset(f, engine="h5netcdf").load()

proj = abi["goes_imager_projection"].attrs                # ABI fixed grid -> latitude and longitude
h = proj["perspective_point_height"]
geos = Proj(proj="geos", h=h, lon_0=proj["longitude_of_projection_origin"], sweep=proj["sweep_angle_axis"])
X, Y = np.meshgrid(abi["x"].values * h, abi["y"].values * h)
abi_lon, abi_lat = geos(X, Y, inverse=True)               # points off the Earth's disk come back as inf
abi_aod = abi["AOD"].where(abi["DQF"] == 0).values         # ABI: high quality only for quantitative use